<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/torneos/notebooks/c6_l7.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C6-L7 · Post-mortem de torneo
12 rondas: Spearman crudo vs neutral, t-stat y veredicto enviar/no-enviar con costos.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/torneos/data/c6_l7.csv'
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c6_l7.csv'), Path('data/c6_l7.csv'), Path('c6_l7.csv')]:
        if cand.exists():
            df = pd.read_csv(cand); break
    print('Fuente: local')
print(df.shape)
print(df.head(5).to_string(index=False))

In [ ]:
import numpy as np
m_c, m_n = df.spearman.mean(), df.spearman_neutral.mean()
sesgo = m_c - m_n
t = m_n / (df.spearman_neutral.std(ddof=1) / np.sqrt(len(df)))
print('crudo %+.4f · neutral %+.4f · sesgo %+.4f · t=%.2f' % (m_c, m_n, sesgo, t))
print('envios:', int((df.decision == 'enviar').sum()), '| descartes:', int((df.decision == 'no_enviar').sum()))

In [ ]:
from scipy.stats import spearmanr
r, _ = spearmanr(df['spearman'], df['spearman_neutral'])
neto_ok = (df[df.decision == 'enviar'].spearman_neutral > 0.01).all()
print('rank-consistencia crudo vs neutral: %.3f | enviados sobre umbral: %s' % (r, neto_ok))
print(df[['ronda', 'spearman_neutral', 'stake', 'decision']].to_string(index=False))

In [ ]:
assert len(df) == 12 and (df.spearman_neutral <= df.spearman + 1e-9).all()
assert abs(df.spearman_neutral.mean() - 0.0181) < 0.003
assert 2.0 < (df.spearman_neutral.mean() / (df.spearman_neutral.std(ddof=1) / np.sqrt(len(df)))) < 2.6
assert (df.decision == 'enviar').sum() == 8 and (df.spearman_neutral[df.decision == 'enviar'] > 0.01).all()
print('OK L7: post-mortem verificado, t > 2 con 8 envios justificados')